## Total Forecasting 

### 초반 설정 (패키지 설치)

In [2]:
%pip install pandas numpy holidays scikit-learn xgboost pymysql sshtunnel python-dotenv openpyxl requests tqdm urllib3 ipykernel 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
!pip uninstall -y paramiko
!pip install paramiko==3.5.1 sshtunnel==0.4.0

Found existing installation: paramiko 3.5.1
Uninstalling paramiko-3.5.1:
  Successfully uninstalled paramiko-3.5.1
Defaulting to user installation because normal site-packages is not writeable
  Using cached paramiko-3.5.1-py3-none-any.whl.metadata (4.6 kB)
Using cached paramiko-3.5.1-py3-none-any.whl (227 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import sys
print(sys.executable)

C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe


In [4]:
import pandas as pd
import pymysql
from sshtunnel import SSHTunnelForwarder
from dotenv import load_dotenv
import requests
import numpy as np
from tqdm import tqdm

print("import 성공")

import 성공


In [3]:
pip install "paramiko<4"

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: paramiko
    Found existing installation: paramiko 4.0.0
    Uninstalling paramiko-4.0.0:
      Successfully uninstalled paramiko-4.0.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Data load & Update

In [2]:
# 최신 데이터 로딩 파이프라인
import pandas as pd
import pymysql
from sshtunnel import SSHTunnelForwarder
import os
import numpy as np
from dotenv import load_dotenv

class AutoContainerGeneration:
    def __init__(self, version=6, debug=False):
        self.version = version
        self.debug = debug
        print(f"AutoContainerGeneration version: {version}")
        load_dotenv()

    # 공통 MySQL 데이터 추출 메서드
    def fetch_data(self, query, ssh_host, ssh_user, ssh_private_key,
                   mysql_host, mysql_port, mysql_user, mysql_password, mysql_database):
        try:
            with SSHTunnelForwarder(
                (ssh_host, 22),
                ssh_username=ssh_user,
                ssh_private_key=ssh_private_key,
                remote_bind_address=(mysql_host, mysql_port)
            ) as tunnel:
                print("SSH 터널 연결 성공")
                with pymysql.connect(
                    host='127.0.0.1', user=mysql_user, passwd=mysql_password,
                    db=mysql_database, charset='utf8', port=tunnel.local_bind_port,
                    cursorclass=pymysql.cursors.DictCursor) as conn:
                    with conn.cursor() as cur:
                        cur.execute(query)
                        results = cur.fetchall()
                        print("쿼리 실행 완료")
                        return pd.DataFrame(results)
        except Exception as e:
            print(f"Error fetching data for: {e}")
            return None

    def fetch_cluster_data(self):
        ssh_host = os.getenv("SSH_HOST_VER_1")
        ssh_user = os.getenv("SSH_USER")
        ssh_private_key = os.getenv(r"SSH_PRIVATE_KEY")
        mysql_host = os.getenv("MYSQL_HOST")
        mysql_port = 3306
        mysql_user = os.getenv("MYSQL_USER")
        mysql_password = os.getenv("MYSQL_PASSWORD")
        mysql_database = os.getenv("MYSQL_DATABASE")

        print(f'Environment: {mysql_host}')

        # 쿼리단에서 기타_당일, 기타_이월까지 분리해서 추출
        cluster_query = """
            SELECT
                hub_cluster_plan.plan_date AS 날짜,
                COUNT(DISTINCT shipping_shippingitem.id) AS "총 물량",
                
                -- [1] 지그재그 당일/이월 분리
                COUNT(DISTINCT CASE 
                    WHEN shop_shop.name = '지그재그 당일배송' AND (
                        JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) = 'sameday' OR 
                        (JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) IS NULL AND HOUR(order_order.`timestamp`) < 13)
                    ) THEN shipping_shippingitem.id END) AS "지그재그_당일",
                    
                COUNT(DISTINCT CASE 
                    WHEN shop_shop.name = '지그재그 당일배송' AND (
                        JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) = 'nextday' OR 
                        (JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) IS NULL AND HOUR(order_order.`timestamp`) >= 13)
                    ) THEN shipping_shippingitem.id END) AS "지그재그_이월",
                    
                -- [2] 기타 셀러 당일/이월 분리 (지그재그 제외)
                COUNT(DISTINCT CASE 
                    WHEN shop_shop.name != '지그재그 당일배송' AND (
                        JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) = 'sameday' OR 
                        (JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) IS NULL AND HOUR(order_order.`timestamp`) < 13)
                    ) THEN shipping_shippingitem.id END) AS "기타_당일",
                    
                COUNT(DISTINCT CASE 
                    WHEN shop_shop.name != '지그재그 당일배송' AND (
                        JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) = 'nextday' OR 
                        (JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) IS NULL AND HOUR(order_order.`timestamp`) >= 13)
                    ) THEN shipping_shippingitem.id END) AS "기타_이월",

                -- [3] 로깅용 데이터
                COUNT(DISTINCT CASE WHEN shop_shop.name = '지그재그 당일배송' THEN shipping_shippingitem.id END) AS "지그재그 당일배송",
                COUNT(DISTINCT CASE WHEN shop_shop.name = '카카오모빌리티' THEN shipping_shippingitem.id END) AS "카카오모빌리티",
                COUNT(DISTINCT CASE WHEN shop_shop.name = '부스터스' THEN shipping_shippingitem.id END) AS "부스터스",
                COUNT(DISTINCT shop_shop.name) AS "셀러수"

            FROM shipping_shippingitem
            JOIN order_order
                ON shipping_shippingitem.order_id = order_order.id
            JOIN hub_cluster_plan_item
                ON shipping_shippingitem.id = hub_cluster_plan_item.shipping_item_id
            JOIN hub_cluster_plan
                ON hub_cluster_plan_item.cluster_plan_id = hub_cluster_plan.id
            JOIN location_address la
                ON shipping_shippingitem.address_id = la.id
            JOIN location_sector ls
                ON shipping_shippingitem.designated_sector_id = ls.id
            JOIN shop_shop
                ON shop_shop.id = order_order.shop_id
            WHERE hub_cluster_plan.plan_date <= CURRENT_DATE()
            AND ls.allowed = 1
            GROUP BY hub_cluster_plan.plan_date
            ORDER BY hub_cluster_plan.plan_date;
                    """
        return self.fetch_data(cluster_query, ssh_host, ssh_user, ssh_private_key,
                               mysql_host, mysql_port, mysql_user, mysql_password, mysql_database)

# ==========================================================
# 2. 실행 및 마스터 규격 병합 파이프라인
# ==========================================================
if __name__ == "__main__":
    generator = AutoContainerGeneration(version=6, debug=True)
    df_forecast = generator.fetch_cluster_data()
    
    if df_forecast is None or df_forecast.empty:
        print("🚨 DB에서 데이터를 가져오지 못했습니다.")
    else:
        print("✅ 최신 DB 3개월치 4-Track 추출 완료")

        db_df = df_forecast.copy()
        
        # 쿼리에서 이미 "총 물량"으로 뽑았으므로, rename 없이 바로 사용하거나 확실하게 공백을 맞춤
        db_df.columns = [c.replace('총_물량', '총 물량') for c in db_df.columns]
        db_df['날짜'] = pd.to_datetime(db_df['날짜'], format='%Y-%m-%d', errors='coerce')

        # 기존 저장된 과거 마스터 데이터 로드
        try:
            base = pd.read_excel("./data/total_latest.xlsx")
            base['날짜'] = pd.to_datetime(base['날짜'], format='%Y-%m-%d', errors='coerce')
        except FileNotFoundError:
            base = pd.DataFrame()

        # 데이터 업데이트 (위아래 결합) -> 3개월치 덮어씌우기
        data = pd.concat([base, db_df], axis=0).reset_index(drop=True)
        data = data.sort_values(by=['날짜'])
        data = data.drop_duplicates(subset=['날짜'], keep='last').reset_index(drop=True)

        # 엑셀 저장 전 최종 정리!
        final_cols = [
            '날짜', '총 물량', 
            '지그재그_당일', '지그재그_이월', 
            '기타_당일', '기타_이월',
            '지그재그 당일배송', '카카오모빌리티', '부스터스', '셀러수'
        ]
        
        # 필수 컬럼만 남기고, 혹시 비어있는 아주 옛날 데이터(NaN)는 0으로 처리
        data = data[final_cols].fillna(0)

        # 신규 데이터 백업 덮어쓰기
        data.to_excel("./data/total_latest.xlsx", index=False)
        print("✨ 신규 4-Track 데이터 엑셀 저장 완료! 이제 빈칸 없습니다!")
        display(data.tail(5))


# ----------------------------------------------------------------------------------------------------------
# 기온 데이터 API 호출 및 병합

import requests
import pandas as pd
import numpy as np
import urllib3
from tqdm import tqdm
from datetime import datetime, timedelta

# -----------------------
# 0. 공통 설정
# -----------------------
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

serviceKey = "ZjfdmTgpRQm33Zk4KZUJIQ"

# ===== 단기예보 =====
def fetch_short_term_forecast(base_date: str, base_time: str, region_point: dict):
    """단기예보 API 호출 → 날짜별 평균기온 DataFrame 반환"""
    records_all = []

    for region, (nx, ny) in tqdm(region_point.items(), desc="단기예보 수집"):
        url = (
            f"https://apihub.kma.go.kr/api/typ02/openApi/VilageFcstInfoService_2.0/getVilageFcst"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&base_date={base_date}&base_time={base_time}"
            f"&nx={nx}&ny={ny}&authKey={serviceKey}"
        )
        try:
            response = requests.get(url, verify=False, timeout=20)
            response.raise_for_status()
            json_response = response.json()
            items = json_response["response"]["body"]["items"]["item"]
        except Exception as e:
            print(f"⚠️ {region} 단기 데이터 요청 실패: {e}")
            continue

        informations = {}
        for it in items:
            cate = it["category"] #예보 데이터 종류, TMP 기온
            fcstDate = it["fcstDate"] #예보 날짜
            fcstTime = it["fcstTime"] #예보 시간
            fcstValue = it["fcstValue"] 
            key = f"{fcstDate}_{fcstTime}"
            if key not in informations:
                informations[key] = {}
            informations[key][cate] = fcstValue #데이터 저장

        region_records = []
        for key, vals in informations.items():
            if "TMP" in vals:
                date = key.split("_")[0]
                region_records.append({"date": date, "TMP": float(vals["TMP"])}) #날짜별 데이터 생성

        if not region_records:
            continue

        df_region = ( #날짜별 평균 기온 계산
            pd.DataFrame(region_records)
            .groupby("date", as_index=False)["TMP"]
            .mean()
            .rename(columns={"TMP": "avg_temp"})
        )
        df_region["지역"] = region #지역 컬럼 추가
        records_all.append(df_region)

    if not records_all:
        return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

    df_all = pd.concat(records_all, ignore_index=True) #전체 지역 합치기
    df_all.rename(columns={"date": "날짜"}, inplace=True)

    # 날짜 형식 통일
    df_all["날짜"] = pd.to_datetime(df_all["날짜"], format="%Y%m%d").dt.strftime("%Y-%m-%d")
    return df_all

# ===== 중기예보 =====
def fetch_mid_term_forecast(base_date: str, region_code: dict):
    """중기예보 API 호출 → 날짜별 평균기온 DataFrame 반환"""
    records = []
    #지역 이름, 기상청 중기예보 지역 코드
    for region, regId in tqdm(region_code.items(), desc="중기예보 수집"):
        url = (
            f"https://apihub.kma.go.kr/api/typ02/openApi/MidFcstInfoService/getMidTa"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&regId={regId}&tmFc={base_date}0600&authKey={serviceKey}" # 지역 코드, 예보 기준 시간, API키
        )
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            data = response.json()["response"]["body"]["items"]["item"][0]
        except Exception as e:
            print(f"⚠️ {region}({regId}) 중기 데이터 요청 실패: {e}")
            continue

        for day in range(4, 11): #예보 범위 (4일후 ~ 10일후)
            ta_min = data.get(f"taMin{day}") # x일 후 최저기온
            ta_max = data.get(f"taMax{day}") # x일 후 최고기온
            if ta_min is not None and ta_max is not None:
                avg_temp = (ta_min + ta_max) / 2 # 평균 기온 계산 : (최저기온 + 최고기온)/2
                record = { # 리스트에 저장
                    "날짜": (datetime.now() + timedelta(days=day)).strftime("%Y-%m-%d"), #오늘을 기준으로 + day해서 날짜 계산(ex. 3월 11일, 4, 3월 15일)
                    "지역": region,
                    "avg_temp": avg_temp,
                }
                records.append(record)

    return pd.DataFrame(records, columns=["날짜", "지역", "avg_temp"])

# ===== 1) 지역 정의 =====
region_point = { # 지역별 좌표 정의
    '서울특별시': ('61','126'),
    '인천광역시': ('54','125'),
    '경기도': ('60','121'),
    '대전광역시': ('68','100'),
    '충청남도': ('65','110')
}
region_code = { #지역별 중기 예보 코드 정의
    '서울특별시': '11B10101',
    '인천광역시': '11B20201',
    '경기도': '11B20601',
    '대전광역시': '11C20401',
    '충청남도': '11C20101'
}

# ===== 2) 단기/중기 데이터 호출 =====
short_df = fetch_short_term_forecast(base_date=datetime.now().strftime("%Y%m%d"), base_time="0200", region_point=region_point)
mid_df = fetch_mid_term_forecast(base_date=datetime.now().strftime("%Y%m%d"), region_code=region_code)

# ===== 3) 단기 + 중기 통합 및 중복 제거 =====
combined_df = pd.concat([short_df, mid_df], ignore_index=True)

# 날짜 형식 완전 통일
combined_df["날짜"] = pd.to_datetime(combined_df["날짜"]).dt.strftime("%Y-%m-%d")

# 날짜 + 지역 기준 중복 제거 (단기 데이터 우선)
final_df = combined_df.drop_duplicates(subset=["날짜", "지역"], keep="first").reset_index(drop=True)

# 지역 + 날짜 정렬
final_df = final_df.sort_values(by=['지역','날짜']).reset_index(drop=True)

# avg_temp 소수점 첫째 자리 유효
final_df['avg_temp'] = np.round(final_df['avg_temp'], 1)

# 컬럼 순서 정렬
final_df = final_df[['날짜','지역','avg_temp']]

# 기존 기온 데이터 업로드
base_temp = pd.read_excel("./data/recent_temp.xlsx")
print("기존 기온 데이터 업로드 완료")

# 기온 데이터 업데이트
temp_data = pd.concat([base_temp, final_df], axis=0).reset_index(drop=True)
temp_data['날짜'] = pd.to_datetime(temp_data['날짜'], format='%Y-%m-%d', errors='coerce')
print("기온 데이터 업데이트 완료")

# 중복 제거
temp_data = temp_data.drop_duplicates(subset=['날짜','지역'], keep='last').reset_index(drop=True)
print("기온 데이터 중복 제거")

# 신규 기온 데이터 저장
temp_data = temp_data.sort_values(by=['지역','날짜'])
temp_data.to_excel("./data/recent_temp.xlsx", index=False)
print("신규 기온 데이터 저장 완료")

display(temp_data.tail(7))

# 날짜 별 기온 데이터 merge
temp_merge_data = temp_data.groupby('날짜')['avg_temp'].mean().reset_index()
temp_merge_data['avg_temp'] = np.round(temp_merge_data['avg_temp'], 1)
data = pd.merge(data, temp_merge_data, how='left', on='날짜')
print("기온 데이터 병합 완료!")

display(data)


AutoContainerGeneration version: 6
Environment: prod-daas-dw-cluster.cluster-cgrjlsurd8vz.ap-northeast-2.rds.amazonaws.com
SSH 터널 연결 성공
쿼리 실행 완료
✅ 최신 DB 3개월치 4-Track 추출 완료
✨ 신규 4-Track 데이터 엑셀 저장 완료! 이제 빈칸 없습니다!


,날짜,총 물량,지그재그_당일,지그재그_이월,기타_당일,기타_이월,지그재그 당일배송,카카오모빌리티,부스터스,셀러수
856,2026-05-08,18717,7240,7675,3791,11,14915,950,1518,24
857,2026-05-09,11157,5320,5422,415,0,10742,263,0,7
858,2026-05-10,10583,5293,5009,281,0,10302,162,0,7
859,2026-05-11,19305,6459,6647,6142,57,13106,1431,2149,23
860,2026-05-12,17938,6704,7592,3626,16,14296,921,567,24


중기예보 수집: 100%|██████████| 5/5 [00:01<00:00,  2.90it/s]


기존 기온 데이터 업로드 완료
기온 데이터 업데이트 완료
기온 데이터 중복 제거
신규 기온 데이터 저장 완료


,날짜,지역,avg_temp
4343,2026-05-16,충청남도,17.0
4344,2026-05-17,충청남도,20.5
4345,2026-05-18,충청남도,19.5
4346,2026-05-19,충청남도,19.0
4347,2026-05-20,충청남도,19.0
4348,2026-05-21,충청남도,19.5
4349,2026-05-22,충청남도,20.0


기온 데이터 병합 완료!


,날짜,총 물량,지그재그_당일,지그재그_이월,기타_당일,기타_이월,지그재그 당일배송,카카오모빌리티,부스터스,셀러수,avg_temp
0,2024-01-02,3220,0,0,0,0,1857,500,0,5,2.9
1,2024-01-03,3293,0,0,0,0,2392,497,0,5,1.5
2,2024-01-04,3901,0,0,0,0,3026,499,0,5,1.2
3,2024-01-05,3907,0,0,0,0,3145,504,0,4,4.6
4,2024-01-06,2129,0,0,0,0,2126,0,0,1,0.7
...,...,...,...,...,...,...,...,...,...,...,...
856,2026-05-08,18717,7240,7675,3791,11,14915,950,1518,24,13.8
857,2026-05-09,11157,5320,5422,415,0,10742,263,0,7,16.0
858,2026-05-10,10583,5293,5009,281,0,10302,162,0,7,16.9
859,2026-05-11,19305,6459,6647,6142,57,13106,1431,2149,23,17.9


### Feature & Modeling

In [3]:
import numpy as np
import pandas as pd
import holidays
from sklearn.metrics import mean_absolute_percentage_error
from xgboost import XGBRegressor

# =====================================
# 0. v.6 핵심 타겟 정의 (4-Track + Meta)
# =====================================
ZIG_SAME = "지그재그_당일"
ZIG_NEXT = "지그재그_이월"
OTH_SAME = "기타_당일"
OTH_NEXT = "기타_이월"
TOTAL = "총 물량"

BASE_TARGETS = [ZIG_SAME, ZIG_NEXT, OTH_SAME, OTH_NEXT]
ALL_TARGETS = BASE_TARGETS + [TOTAL]

# =====================================
# 피처 엔지니어링 함수 (기존 로직 유지)
# =====================================
def add_temp_shift_features(df, date_col="날짜", temp_col="avg_temp"):
    df = df.copy().sort_values(date_col)
    temp_ma10 = df[temp_col].rolling(7, min_periods=1).min()
    diff = temp_ma10 - df[temp_col]
    df["temp_shift"] = diff.abs()
    df["temp_shift_sign"] = np.sign(diff)
    df[["temp_shift", "temp_shift_sign"]] = df[["temp_shift", "temp_shift_sign"]].fillna(0.0)
    return df

def add_target_lag_rolling_features(df, target_col, lags=(1, 7, 14), roll_windows=(7, 14)):
    df = df.copy().sort_values("날짜").reset_index(drop=True)
    if target_col not in df.columns:
        for lag in lags: df[f"{target_col}_lag_{lag}"] = np.nan
        for w in roll_windows:
            df[f"{target_col}_roll_mean_{w}"] = np.nan
            df[f"{target_col}_roll_std_{w}"] = np.nan
        return df

    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = df[target_col].shift(lag)

    shifted = df[target_col].shift(1)
    for w in roll_windows:
        df[f"{target_col}_roll_mean_{w}"] = shifted.rolling(w, min_periods=max(2, w // 4)).mean()
        df[f"{target_col}_roll_std_{w}"] = shifted.rolling(w, min_periods=max(2, w // 4)).std()
    return df

def preprocess_features(df, manual_spikes_dict=None):
    df = df.copy()
    df["날짜"] = pd.to_datetime(df["날짜"]).dt.normalize()
    df = df.sort_values("날짜").reset_index(drop=True)

    # 1. 기본 날짜 관련 피처
    df["month"] = df["날짜"].dt.month
    df["dow"] = df["날짜"].dt.weekday
    df["weekend"] = df["dow"].apply(lambda x: 1 if x in [5, 6] else 0)
    df["week_of_year"] = df["날짜"].apply(lambda x: x.isocalendar().week)

    df["month"] = pd.Categorical(df["month"], categories=list(range(1, 13)))
    df["dow"] = pd.Categorical(df["dow"], categories=list(range(0, 7)))

    month_dummies = pd.get_dummies(df["month"], prefix="month").astype(int)
    dow_dummies = pd.get_dummies(df["dow"], prefix="dow").astype(int)
    df = pd.concat([df, month_dummies, dow_dummies], axis=1)

    df["month"] = df["month"].astype(int)
    df["dow"] = df["dow"].astype(int)

    # 2. 이벤트 기간
    event_periods = [
        ("2024-03-25", "2024-04-07"), ("2024-06-10", "2024-06-17"),
        ("2024-06-24", "2024-07-07"), ("2024-09-23", "2024-10-06"),
        ("2024-11-18", "2024-12-01"), ("2025-03-24", "2025-04-07"),
        ("2025-06-01", "2025-06-30"), ("2025-09-22", "2025-10-05"),
        ("2025-10-21", "2025-10-21"), ("2025-11-17", "2025-12-02"),
        ("2026-03-23", "2026-04-06"), 
    ]
    df["is_event"] = 0
    for start, end in event_periods:
        mask = (df["날짜"] >= pd.to_datetime(start)) & (df["날짜"] <= pd.to_datetime(end))
        df.loc[mask, "is_event"] = 1

    # 3. 자동 Spike 탐지
    auto_cols = [c for c in ALL_TARGETS if c in df.columns]

    for col in auto_cols:
        df[f"{col}_spike_up_mild"] = 0
        df[f"{col}_spike_up_strong"] = 0
        df[f"{col}_spike_down"] = 0

        trend_col, resid_col = f"{col}_trend", f"{col}_resid_ratio"
        df[trend_col], df[resid_col] = np.nan, np.nan

        for d in range(7):
            idx_d = df.index[df["dow"] == d]
            if len(idx_d) == 0: continue
            s = df.loc[idx_d, col].sort_index()
            baseline = s.shift(1).rolling(window=3, min_periods=2).mean()
            
            # ⭐️ 해결: 뒤에 .values를 붙여 순수 배열(Array) 형태로 집어넣음
            df.loc[idx_d, trend_col] = baseline.astype(float).values
            df.loc[idx_d, resid_col] = ((s - baseline) / (baseline + 1e-5)).astype(float).values

        for wk_flag in [0, 1]:
            idx = df.index[(df["weekend"] == wk_flag) & df[resid_col].notnull()]
            if len(idx) == 0: continue
            upper1, upper2, lower1 = (0.10, 0.20, -0.15) if wk_flag == 0 else (0.20, 0.25, -0.10)

            mask_up_mild = (df[resid_col] >= upper1) & (df[resid_col] < upper2) & (df["weekend"] == wk_flag)
            mask_up_strong = (df[resid_col] >= upper2) & (df["weekend"] == wk_flag)
            mask_down = (df[resid_col] <= lower1) & (df["weekend"] == wk_flag)

            df.loc[mask_up_mild, f"{col}_spike_up_mild"] = 1
            df.loc[mask_up_strong, f"{col}_spike_up_strong"] = 1
            df.loc[mask_down, f"{col}_spike_down"] = 1

    # 4. 수동 Spike 지정
    if manual_spikes_dict:
        for tgt, spikes in manual_spikes_dict.items():
            for spike_type, dates in spikes.items(): 
                if dates:
                    dts = pd.to_datetime(dates).normalize()
                    df.loc[df["날짜"].isin(dts), f"{tgt}_spike_{spike_type}"] = 1

    # 5. 공휴일 피크 및 기온 피처
    years = sorted(df["날짜"].dt.year.unique().tolist())
    kr_holidays = holidays.KR(years=years)
    df["holiday"] = df["날짜"].isin(kr_holidays).astype(int)
    df.loc[df["weekend"] == 1, "holiday"] = 0

    exclude_dates = pd.to_datetime(["2024-02-09", "2024-02-10", "2025-01-28", "2026-02-16"]).normalize()
    df.loc[df['날짜'].isin(exclude_dates), "holiday"] = 0

    if "avg_temp" in df.columns:
        df = add_temp_shift_features(df, temp_col="avg_temp")
        df = df.drop(columns=["avg_temp"], errors="ignore")

    # 6. ALL_TARGETS 각각 lag/rolling 생성
    for tgt in ALL_TARGETS:
        df = add_target_lag_rolling_features(df, target_col=tgt, lags=(1, 7, 14, 21, 28), roll_windows=(7, 14, 21, 28))

    # 7. 불필요 컬럼 드랍
    drop_cols = [f"{c}_trend" for c in auto_cols] + [f"{c}_resid_ratio" for c in auto_cols]
    df = df.drop(columns=drop_cols, errors="ignore")
    return df

# =====================================
#  XGBoost Helper 및 Loss 함수 
# =====================================
def _safe_best_iteration(xgb_model):
    if hasattr(xgb_model, 'best_iteration') and xgb_model.best_iteration is not None:
        return xgb_model.best_iteration
    return xgb_model.n_estimators - 1

def _xgb_predict_with_best_iteration(model, X):
    iteration = _safe_best_iteration(model)
    return model.predict(X, iteration_range=(0, iteration + 1))

def weighted_mse(alpha=1):
    def weighted_mse_fixed(label, pred):
        residual = (label - pred).astype("float")
        grad = np.where(residual > 0, -2 * alpha * residual, -2 * residual)
        hess = np.where(residual > 0, 2 * alpha, 2.0)
        return grad, hess
    return weighted_mse_fixed

# =====================================
# v.6 학습/검증/예측 (4 Base + 1 Meta Chain Inference)
# =====================================
def train_validate_forecast(
    train,
    forecast_days=7,
    valid_days=7,
    temp_future_df=None,
    manual_spikes_train=None,
    manual_spikes_future=None,
    base_recent_window=150,
    meta_recent_window=240,
    meta_blend_weight=0.2,
):
    RANDOM_STATE = 42
    MIN_TRAIN_SAMPLES = 30
    ZERO_RUN_MIN = 3

    def _ensure_cols(df, cols, fill_value=0.0):
        out = df.copy()
        missing = [c for c in cols if c not in out.columns]
        for c in missing:
            out[c] = fill_value
        return out[cols]

    def _valid_target_mask(df, target_col, zero_run_min=3):
        s = pd.to_numeric(df[target_col], errors="coerce")

        base_mask = s.notna() & (s > 0)

        is_zero = s.fillna(0).eq(0)
        group_id = is_zero.ne(is_zero.shift()).cumsum()
        zero_run_len = is_zero.groupby(group_id).transform("sum")
        long_zero_mask = is_zero & (zero_run_len >= zero_run_min)

        return base_mask & (~long_zero_mask)

    def _clean_lag_features_from_fake_zero(df, target_cols, zero_run_min=3):
        out = df.copy()

        for col in target_cols:
            if col not in out.columns:
                continue

            s = pd.to_numeric(out[col], errors="coerce")
            is_zero = s.fillna(0).eq(0)
            group_id = is_zero.ne(is_zero.shift()).cumsum()
            zero_run_len = is_zero.groupby(group_id).transform("sum")

            fake_zero_mask = is_zero & (zero_run_len >= zero_run_min)
            out.loc[fake_zero_mask, col] = np.nan

        return out

    def _add_event_day_features(df):
        out = df.copy()

        if "is_event" not in out.columns:
            out["is_event"] = 0

        out["event_day_index"] = 0
        out["event_block_id"] = 0

        is_event = out["is_event"].fillna(0).astype(int).eq(1)
        block_id = is_event.ne(is_event.shift()).cumsum()

        out.loc[is_event, "event_block_id"] = block_id[is_event].astype(int)
        out.loc[is_event, "event_day_index"] = (
            out[is_event]
            .groupby(block_id[is_event])
            .cumcount()
            + 1
        )

        out["event_after_1d"] = out["is_event"].shift(1).fillna(0).astype(int)
        out["event_after_2d"] = out["is_event"].shift(2).fillna(0).astype(int)
        out["event_before_1d"] = out["is_event"].shift(-1).fillna(0).astype(int)

        return out

    def weighted_mse_alpha(alpha=1.5):
        def obj(label, pred):
            residual = label - pred
            grad = np.where(residual > 0, -2 * alpha * residual, -2 * residual)
            hess = np.where(residual > 0, 2 * alpha, 2.0)
            return grad, hess

        return obj

    train_sorted = train.copy()
    train_sorted["날짜"] = pd.to_datetime(train_sorted["날짜"]).dt.normalize()
    train_sorted = train_sorted.sort_values("날짜").reset_index(drop=True)

    train_for_feature = _clean_lag_features_from_fake_zero(
        train_sorted,
        target_cols=BASE_TARGETS,
        zero_run_min=ZERO_RUN_MIN,
    )

    train_feat_all = preprocess_features(
        train_for_feature,
        manual_spikes_dict=manual_spikes_train,
    )

    train_feat_all = _add_event_day_features(train_feat_all)

    split_idx = len(train_feat_all) - valid_days
    train_df = train_feat_all.iloc[:split_idx].copy()
    valid_df = train_feat_all.iloc[split_idx:].copy()
    full_df = train_feat_all.copy()

    last_date = train_sorted["날짜"].max()
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1),
        periods=forecast_days,
    )
    future_df = pd.DataFrame({"날짜": future_dates})

    if temp_future_df is not None:
        temp_future_df = temp_future_df.copy()
        temp_future_df["날짜"] = pd.to_datetime(temp_future_df["날짜"]).dt.normalize()

        future_temp_input = pd.merge(
            future_df,
            temp_future_df,
            on="날짜",
            how="left",
        )

        future_temp_input["avg_temp"] = (
            future_temp_input["avg_temp"]
            .ffill()
            .bfill()
            .fillna(15.0)
        )
    else:
        future_temp_input = future_df.copy()
        future_temp_input["avg_temp"] = 15.0

    future_feat_df = preprocess_features(
        future_temp_input,
        manual_spikes_dict=manual_spikes_future,
    )

    future_feat_df = _add_event_day_features(future_feat_df)
    future_feat_df = future_feat_df.iloc[-forecast_days:].reset_index(drop=True)

    for tgt in ALL_TARGETS:
        if tgt not in future_feat_df.columns:
            future_feat_df[tgt] = np.nan

    all_cols = full_df.columns.tolist()
    features_dict = {}

    LEAKAGE_COLS = [
        "지그재그 당일배송",
        "카카오모빌리티",
        "부스터스",
        "other_seller",
        "셀러수",
        "이월_물량",
        "당일_물량",
        "기타_물량",
        "기타_총물량",
    ]

    for tgt in BASE_TARGETS:
        drop_base = ["날짜"] + ALL_TARGETS + LEAKAGE_COLS

        exclude_others = [
            c for c in all_cols
            if any(
                c.startswith(f"{other}_")
                for other in ALL_TARGETS
                if other != tgt
            )
        ]

        features_dict[tgt] = [
            c for c in all_cols
            if c not in drop_base + exclude_others
        ]

    drop_meta = ["날짜", TOTAL] + LEAKAGE_COLS + BASE_TARGETS

    features_dict[TOTAL] = [
        c for c in all_cols
        if c not in drop_meta
    ]

    xgb_base_params = {
        "n_estimators": 900,
        "learning_rate": 0.015,
        "max_depth": 3,
        "min_child_weight": 5,
        "gamma": 0.05,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.1,
        "reg_lambda": 3.0,
        "objective": weighted_mse_alpha(alpha=1.6),
        "random_state": RANDOM_STATE,
        "tree_method": "hist",
        "eval_metric": "rmse",
    }

    xgb_meta_params = {
        "n_estimators": 500,
        "learning_rate": 0.02,
        "max_depth": 2,
        "min_child_weight": 8,
        "gamma": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.2,
        "reg_lambda": 5.0,
        "objective": weighted_mse_alpha(alpha=1.3),
        "random_state": RANDOM_STATE,
        "tree_method": "hist",
        "eval_metric": "rmse",
    }

    trained_models = {}
    pred_train_base = {}
    pred_valid_base = {}
    pred_full_base = {}

    print("\n=== 학습 제외 데이터 확인 ===")

    for tgt in BASE_TARGETS:
        train_recent = train_df.tail(base_recent_window).copy()
        full_recent = full_df.tail(base_recent_window).copy()

        X_tr = _ensure_cols(train_recent, features_dict[tgt]).fillna(0.0)
        y_tr = pd.to_numeric(train_recent[tgt], errors="coerce")

        X_val = _ensure_cols(valid_df, features_dict[tgt]).fillna(0.0)

        X_full_recent = _ensure_cols(full_recent, features_dict[tgt]).fillna(0.0)
        y_full_recent = pd.to_numeric(full_recent[tgt], errors="coerce")

        X_train_all = _ensure_cols(train_df, features_dict[tgt]).fillna(0.0)
        X_full_all = _ensure_cols(full_df, features_dict[tgt]).fillna(0.0)

        mask_tr = _valid_target_mask(train_recent, tgt, ZERO_RUN_MIN)
        mask_full = _valid_target_mask(full_recent, tgt, ZERO_RUN_MIN)

        print(
            f"{tgt}: train 최근 {len(train_recent)}건 중 사용 {mask_tr.sum()}건 / 제외 {len(mask_tr) - mask_tr.sum()}건, "
            f"full 최근 {len(full_recent)}건 중 사용 {mask_full.sum()}건 / 제외 {len(mask_full) - mask_full.sum()}건"
        )

        if mask_tr.sum() >= MIN_TRAIN_SAMPLES:
            model = XGBRegressor(**xgb_base_params)
            model.fit(
                X_tr.loc[mask_tr],
                np.log1p(y_tr.loc[mask_tr].astype(float)),
                verbose=False,
            )

            pred_train_base[tgt] = np.clip(
                np.expm1(model.predict(X_train_all)),
                0,
                None,
            )
            pred_valid_base[tgt] = np.clip(
                np.expm1(model.predict(X_val)),
                0,
                None,
            )

            model_full = XGBRegressor(**xgb_base_params)
            model_full.fit(
                X_full_recent.loc[mask_full],
                np.log1p(y_full_recent.loc[mask_full].astype(float)),
                verbose=False,
            )

            pred_full_base[tgt] = np.clip(
                np.expm1(model_full.predict(X_full_all)),
                0,
                None,
            )

            trained_models[tgt] = model_full

        else:
            recent_valid = y_full_recent.loc[mask_full].tail(28)
            mean_val = float(recent_valid.mean()) if len(recent_valid) > 0 else 0.0

            pred_train_base[tgt] = np.full(len(X_train_all), mean_val)
            pred_valid_base[tgt] = np.full(len(X_val), mean_val)
            pred_full_base[tgt] = np.full(len(X_full_all), mean_val)
            trained_models[tgt] = None

    meta_features = features_dict[TOTAL] + [f"pred_{t}" for t in BASE_TARGETS] + ["base_sum_pred"]

    X_tr_meta = _ensure_cols(train_df, features_dict[TOTAL]).fillna(0.0)
    X_val_meta = _ensure_cols(valid_df, features_dict[TOTAL]).fillna(0.0)
    X_full_meta = _ensure_cols(full_df, features_dict[TOTAL]).fillna(0.0)

    for tgt in BASE_TARGETS:
        X_tr_meta[f"pred_{tgt}"] = pred_train_base[tgt]
        X_val_meta[f"pred_{tgt}"] = pred_valid_base[tgt]
        X_full_meta[f"pred_{tgt}"] = pred_full_base[tgt]

    X_tr_meta["base_sum_pred"] = sum(X_tr_meta[f"pred_{t}"] for t in BASE_TARGETS)
    X_val_meta["base_sum_pred"] = sum(X_val_meta[f"pred_{t}"] for t in BASE_TARGETS)
    X_full_meta["base_sum_pred"] = sum(X_full_meta[f"pred_{t}"] for t in BASE_TARGETS)

    X_tr_meta = X_tr_meta[meta_features]
    X_val_meta = X_val_meta[meta_features]
    X_full_meta = X_full_meta[meta_features]

    y_tr_meta = pd.to_numeric(train_df[TOTAL], errors="coerce")
    y_full_meta = pd.to_numeric(full_df[TOTAL], errors="coerce")

    train_meta_recent_idx = train_df.tail(meta_recent_window).index
    full_meta_recent_idx = full_df.tail(meta_recent_window).index

    X_tr_meta_recent = X_tr_meta.loc[train_meta_recent_idx]
    y_tr_meta_recent = y_tr_meta.loc[train_meta_recent_idx]
    train_df_meta_recent = train_df.loc[train_meta_recent_idx]

    X_full_meta_recent = X_full_meta.loc[full_meta_recent_idx]
    y_full_meta_recent = y_full_meta.loc[full_meta_recent_idx]
    full_df_meta_recent = full_df.loc[full_meta_recent_idx]

    mask_tr_meta = _valid_target_mask(train_df_meta_recent, TOTAL, ZERO_RUN_MIN)
    mask_full_meta = _valid_target_mask(full_df_meta_recent, TOTAL, ZERO_RUN_MIN)

    print(
        f"{TOTAL}: meta 최근 {len(train_df_meta_recent)}건 중 사용 {mask_tr_meta.sum()}건 / 제외 {len(mask_tr_meta) - mask_tr_meta.sum()}건"
    )
    print("========================\n")

    model_meta = XGBRegressor(**xgb_meta_params)
    model_meta.fit(
        X_tr_meta_recent.loc[mask_tr_meta],
        np.log1p(y_tr_meta_recent.loc[mask_tr_meta].astype(float)),
        verbose=False,
    )

    meta_valid_pred = np.clip(
        np.expm1(model_meta.predict(X_val_meta)),
        0,
        None,
    )

    base_sum_valid_pred = X_val_meta["base_sum_pred"].values

    pred_valid_total = (
        (1 - meta_blend_weight) * base_sum_valid_pred
        + meta_blend_weight * meta_valid_pred
    )

    model_meta_full = XGBRegressor(**xgb_meta_params)
    model_meta_full.fit(
        X_full_meta_recent.loc[mask_full_meta],
        np.log1p(y_full_meta_recent.loc[mask_full_meta].astype(float)),
        verbose=False,
    )

    trained_models[TOTAL] = model_meta_full

    print("\n=== 검증 성능 (MAPE) ===")

    actuals_tot = pd.to_numeric(valid_df[TOTAL], errors="coerce").values
    mask_tot = actuals_tot > 0

    if mask_tot.sum() > 0:
        mape_tot = mean_absolute_percentage_error(
            actuals_tot[mask_tot],
            pred_valid_total[mask_tot],
        )
        print(f"총 물량: {mape_tot:.4f} (정확도 {100 * (1 - mape_tot):.2f}%)")

    for tgt in BASE_TARGETS:
        actuals = pd.to_numeric(valid_df[tgt], errors="coerce").values
        preds = pred_valid_base[tgt]
        mask = actuals > 0

        if mask.sum() > 0:
            mape = mean_absolute_percentage_error(actuals[mask], preds[mask])
            print(f"{tgt}: {mape:.4f} (정확도 {100 * (1 - mape):.2f}%)")

    print("========================\n")

    def _overwrite_ts_features(row_df, tgt, buf):
        row = row_df.copy()

        clean_buf = [
            float(x) for x in buf
            if pd.notna(x) and float(x) > 0
        ]

        for lag in (1, 7, 14, 21, 28):
            row[f"{tgt}_lag_{lag}"] = clean_buf[-lag] if len(clean_buf) >= lag else 0.0

        for w in (7, 14, 21, 28):
            tail = clean_buf[-w:]

            row[f"{tgt}_roll_mean_{w}"] = (
                float(np.mean(tail)) if len(tail) >= 2 else 0.0
            )
            row[f"{tgt}_roll_std_{w}"] = (
                float(np.std(tail, ddof=1)) if len(tail) >= 2 else 0.0
            )

        return row

    bufs = {
        tgt: pd.to_numeric(full_df[tgt], errors="coerce").tolist()
        for tgt in ALL_TARGETS
    }

    future_preds = {tgt: [] for tgt in ALL_TARGETS}

    for i in range(forecast_days):
        row = future_feat_df.iloc[[i]].copy()

        for tgt in BASE_TARGETS:
            row = _overwrite_ts_features(row, tgt, bufs[tgt])
            X_row = _ensure_cols(row, features_dict[tgt]).fillna(0.0)

            if trained_models[tgt] is not None:
                pred_i = max(
                    0.0,
                    float(np.expm1(trained_models[tgt].predict(X_row))[0]),
                )
            else:
                recent_buf = [
                    x for x in bufs[tgt]
                    if pd.notna(x) and float(x) > 0
                ]
                pred_i = float(np.mean(recent_buf[-7:])) if len(recent_buf) > 0 else 0.0

            future_preds[tgt].append(pred_i)
            bufs[tgt].append(pred_i)
            row[f"pred_{tgt}"] = pred_i

        base_sum_pred = sum(row[f"pred_{t}"].iloc[0] for t in BASE_TARGETS)
        row["base_sum_pred"] = base_sum_pred

        row = _overwrite_ts_features(row, TOTAL, bufs[TOTAL])
        X_row_meta = _ensure_cols(row, meta_features).fillna(0.0)

        meta_pred = max(
            0.0,
            float(np.expm1(trained_models[TOTAL].predict(X_row_meta))[0]),
        )

        pred_total = (
            (1 - meta_blend_weight) * base_sum_pred
            + meta_blend_weight * meta_pred
        )

        future_preds[TOTAL].append(pred_total)
        bufs[TOTAL].append(pred_total)

    valid_result_df = pd.DataFrame({
        "날짜": valid_df["날짜"].dt.strftime("%Y-%m-%d"),
    })

    valid_result_df["실제_총물량"] = pd.to_numeric(
        valid_df[TOTAL],
        errors="coerce",
    ).values

    valid_result_df["예측_총물량"] = np.round(pred_valid_total).astype(int)
    valid_result_df["base_sum_예측"] = np.round(base_sum_valid_pred).astype(int)
    valid_result_df["meta_예측"] = np.round(meta_valid_pred).astype(int)

    for tgt in BASE_TARGETS:
        valid_result_df[f"실제_{tgt}"] = pd.to_numeric(
            valid_df[tgt],
            errors="coerce",
        ).values
        valid_result_df[f"예측_{tgt}"] = np.round(pred_valid_base[tgt]).astype(int)

    result_df = pd.DataFrame({
        "날짜": future_feat_df["날짜"].dt.strftime("%Y-%m-%d"),
    })

    result_df["총 물량"] = np.round(future_preds[TOTAL]).astype(int)

    for tgt in BASE_TARGETS:
        result_df[tgt] = np.round(future_preds[tgt]).astype(int)

    result_df["base_sum"] = (
        result_df[ZIG_SAME]
        + result_df[ZIG_NEXT]
        + result_df[OTH_SAME]
        + result_df[OTH_NEXT]
    )

    return trained_models, result_df, valid_result_df

### Prediction

In [4]:
# 날짜 형식 정리
data["날짜"] = pd.to_datetime(data["날짜"]).dt.normalize()
temp_merge_data["날짜"] = pd.to_datetime(temp_merge_data["날짜"]).dt.normalize()

trained_models, forecast_df, valid_result = train_validate_forecast(
    train=data,              # ✅ preprocess_features(data) 말고 원본 data
    forecast_days=7,
    valid_days=7,
    temp_future_df=temp_merge_data
)

print("\n🔍 [최근 7일 검증 셋 결과 비교표]")
display(valid_result)

print("\n🚀 [미래 7일 예측 결과표]")
display(forecast_df)


=== 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 96건 / 제외 54건, full 최근 150건 중 사용 103건 / 제외 47건
지그재그_이월: train 최근 150건 중 사용 96건 / 제외 54건, full 최근 150건 중 사용 103건 / 제외 47건
기타_당일: train 최근 150건 중 사용 23건 / 제외 127건, full 최근 150건 중 사용 30건 / 제외 120건
기타_이월: train 최근 150건 중 사용 11건 / 제외 139건, full 최근 150건 중 사용 16건 / 제외 134건
총 물량: meta 최근 240건 중 사용 240건 / 제외 0건


=== 검증 성능 (MAPE) ===
총 물량: 0.1441 (정확도 85.59%)
지그재그_당일: 0.0433 (정확도 95.67%)
지그재그_이월: 0.0680 (정확도 93.20%)
기타_당일: 2.4467 (정확도 -144.67%)
기타_이월: 0.5428 (정확도 45.72%)


🔍 [최근 7일 검증 셋 결과 비교표]


,날짜,실제_총물량,예측_총물량,base_sum_예측,meta_예측,실제_지그재그_당일,예측_지그재그_당일,실제_지그재그_이월,예측_지그재그_이월,실제_기타_당일,예측_기타_당일,실제_기타_이월,예측_기타_이월
854,2026-05-06,17487,15702,15102,18105,5999.0,5892,6587.0,6282,4865.0,2905,36.0,22
855,2026-05-07,19576,17033,16748,18174,6835.0,6854,7676.0,6966,5048.0,2905,17.0,22
856,2026-05-08,18717,17838,18386,15643,7240.0,7490,7675.0,7968,3791.0,2905,11.0,22
857,2026-05-09,11157,14047,14433,12500,5320.0,6318,5422.0,5188,415.0,2905,0.0,22
858,2026-05-10,10583,13245,14057,9996,5293.0,5412,5009.0,5718,281.0,2905,0.0,22
859,2026-05-11,19305,16959,16673,18105,6459.0,6691,6647.0,7054,6142.0,2905,57.0,22
860,2026-05-12,17938,16186,16833,13596,6704.0,6718,7592.0,7187,3626.0,2905,16.0,22



🚀 [미래 7일 예측 결과표]


,날짜,총 물량,지그재그_당일,지그재그_이월,기타_당일,기타_이월,base_sum
0,2026-05-13,18182,6961,7645,3453,31,18090
1,2026-05-14,19042,7532,8273,3251,24,19080
2,2026-05-15,18182,7170,7856,2994,27,18047
3,2026-05-16,13959,5731,5946,2880,26,14583
4,2026-05-17,14120,5831,5657,3232,28,14748
5,2026-05-18,17746,6623,7114,3654,30,17421
6,2026-05-19,17679,6814,7578,3299,26,17717


아래는 백테스팅

In [ ]:
# ==========================================================
# 🕒 백테스팅: 2026-04-21로 돌아가서 4.22 ~ 4.28 예측해보기
# ==========================================================
cutoff_date = pd.to_datetime("2026-04-21")
end_date = pd.to_datetime("2026-04-28")

data["날짜"] = pd.to_datetime(data["날짜"]).dt.normalize()

# 1. 4월 21일까지만 학습
simulated_data = data[data["날짜"] <= cutoff_date].copy()

# 2. 4.22 ~ 4.28 실제 기온만 사용
future_temp_sim = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "avg_temp"]].copy()

# 3. 모델 실행
print(f"🕒 타임머신 작동: {cutoff_date.strftime('%Y-%m-%d')} 기준으로 학습을 시작합니다...\n")

trained_models_sim, forecast_df_sim, valid_result_sim = train_validate_forecast(
    train=simulated_data,        # ✅ preprocess_features() 넣지 않기
    forecast_days=7,
    valid_days=7,
    temp_future_df=future_temp_sim,
    manual_spikes_train=None,
    manual_spikes_future=None
)

# 4. 실제 정답 가져오기
actual_future = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "총 물량"]].copy()

actual_future.rename(columns={"총 물량": "실제_정답"}, inplace=True)
actual_future["날짜"] = actual_future["날짜"].dt.strftime("%Y-%m-%d")

# 5. 예측값과 결합
comparison_df = pd.merge(
    forecast_df_sim[["날짜", "총 물량"]],
    actual_future,
    on="날짜",
    how="left"
)

comparison_df.rename(columns={"총 물량": "모델_예측값"}, inplace=True)

# 6. 오차율 계산
comparison_df["오차율(%)"] = (
    np.abs(comparison_df["실제_정답"] - comparison_df["모델_예측값"])
    / comparison_df["실제_정답"]
    * 100
).round(2)

comparison_df["정확도(%)"] = (100 - comparison_df["오차율(%)"]).round(2)
comparison_df["차이(실제-예측)"] = comparison_df["실제_정답"] - comparison_df["모델_예측값"]

print("\n🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)")
display(comparison_df)

mean_error = comparison_df["오차율(%)"].mean()
print(f"👉 이 기간의 평균 예측 오차율은 {mean_error:.2f}% (정확도 {100 - mean_error:.2f}%) 입니다!")

🕒 타임머신 작동: 2026-04-21 기준으로 학습을 시작합니다...


=== 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 75건 / 제외 75건, full 최근 150건 중 사용 82건 / 제외 68건
지그재그_이월: train 최근 150건 중 사용 75건 / 제외 75건, full 최근 150건 중 사용 82건 / 제외 68건
기타_당일: train 최근 150건 중 사용 74건 / 제외 76건, full 최근 150건 중 사용 81건 / 제외 69건
기타_이월: train 최근 150건 중 사용 46건 / 제외 104건, full 최근 150건 중 사용 51건 / 제외 99건
총 물량: meta 최근 240건 중 사용 240건 / 제외 0건


=== 검증 성능 (MAPE) ===
총 물량: 0.1095 (정확도 89.05%)
지그재그_당일: 0.1037 (정확도 89.63%)
지그재그_이월: 0.0669 (정확도 93.31%)
기타_당일: 0.3155 (정확도 68.45%)
기타_이월: 0.3118 (정확도 68.82%)


🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)


,날짜,모델_예측값,실제_정답,오차율(%),정확도(%),차이(실제-예측)
0,2026-04-22,20315,17814,14.04,85.96,-2501
1,2026-04-23,21189,19861,6.69,93.31,-1328
2,2026-04-24,20552,20536,0.08,99.92,-16
3,2026-04-25,13391,12325,8.65,91.35,-1066
4,2026-04-26,13173,12676,3.92,96.08,-497
5,2026-04-27,20903,21298,1.85,98.15,395
6,2026-04-28,20108,19145,5.03,94.97,-963


👉 이 기간의 평균 예측 오차율은 5.75% (정확도 94.25%) 입니다!


In [58]:
# ==========================================================
# 🕒 백테스팅: 2026-04-21로 돌아가서 4.22 ~ 4.28 예측해보기
# ==========================================================
cutoff_date = pd.to_datetime("2026-04-14")
end_date = pd.to_datetime("2026-04-21")

data["날짜"] = pd.to_datetime(data["날짜"]).dt.normalize()

# 1. 4월 21일까지만 학습
simulated_data = data[data["날짜"] <= cutoff_date].copy()

# 2. 4.22 ~ 4.28 실제 기온만 사용
future_temp_sim = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "avg_temp"]].copy()

# 3. 모델 실행
print(f"🕒 타임머신 작동: {cutoff_date.strftime('%Y-%m-%d')} 기준으로 학습을 시작합니다...\n")

trained_models_sim, forecast_df_sim, valid_result_sim = train_validate_forecast(
    train=simulated_data,        # ✅ preprocess_features() 넣지 않기
    forecast_days=7,
    valid_days=7,
    temp_future_df=future_temp_sim,
    manual_spikes_train=None,
    manual_spikes_future=None
)

# 4. 실제 정답 가져오기
actual_future = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "총 물량"]].copy()

actual_future.rename(columns={"총 물량": "실제_정답"}, inplace=True)
actual_future["날짜"] = actual_future["날짜"].dt.strftime("%Y-%m-%d")

# 5. 예측값과 결합
comparison_df = pd.merge(
    forecast_df_sim[["날짜", "총 물량"]],
    actual_future,
    on="날짜",
    how="left"
)

comparison_df.rename(columns={"총 물량": "모델_예측값"}, inplace=True)

# 6. 오차율 계산
comparison_df["오차율(%)"] = (
    np.abs(comparison_df["실제_정답"] - comparison_df["모델_예측값"])
    / comparison_df["실제_정답"]
    * 100
).round(2)

comparison_df["정확도(%)"] = (100 - comparison_df["오차율(%)"]).round(2)
comparison_df["차이(실제-예측)"] = comparison_df["실제_정답"] - comparison_df["모델_예측값"]

print("\n🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)")
display(comparison_df)

mean_error = comparison_df["오차율(%)"].mean()
print(f"👉 이 기간의 평균 예측 오차율은 {mean_error:.2f}% (정확도 {100 - mean_error:.2f}%) 입니다!")

🕒 타임머신 작동: 2026-04-14 기준으로 학습을 시작합니다...


=== 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 68건 / 제외 82건, full 최근 150건 중 사용 75건 / 제외 75건
지그재그_이월: train 최근 150건 중 사용 68건 / 제외 82건, full 최근 150건 중 사용 75건 / 제외 75건
기타_당일: train 최근 150건 중 사용 67건 / 제외 83건, full 최근 150건 중 사용 74건 / 제외 76건
기타_이월: train 최근 150건 중 사용 42건 / 제외 108건, full 최근 150건 중 사용 46건 / 제외 104건
총 물량: meta 최근 240건 중 사용 240건 / 제외 0건


=== 검증 성능 (MAPE) ===
총 물량: 0.0849 (정확도 91.51%)
지그재그_당일: 0.0867 (정확도 91.33%)
지그재그_이월: 0.0544 (정확도 94.56%)
기타_당일: 0.3405 (정확도 65.95%)
기타_이월: 1.0161 (정확도 -1.61%)


🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)


,날짜,모델_예측값,실제_정답,오차율(%),정확도(%),차이(실제-예측)
0,2026-04-15,18490,20900,11.53,88.47,2410
1,2026-04-16,19808,24131,17.91,82.09,4323
2,2026-04-17,19241,22511,14.53,85.47,3270
3,2026-04-18,12687,12410,2.23,97.77,-277
4,2026-04-19,12190,12215,0.20,99.80,25
5,2026-04-20,16275,21091,22.83,77.17,4816
6,2026-04-21,19718,17805,10.74,89.26,-1913


👉 이 기간의 평균 예측 오차율은 11.42% (정확도 88.58%) 입니다!


In [59]:
# ==========================================================
# 🕒 백테스팅: 2026-04-21로 돌아가서 4.22 ~ 4.28 예측해보기
# ==========================================================
cutoff_date = pd.to_datetime("2026-04-07")
end_date = pd.to_datetime("2026-04-14")

data["날짜"] = pd.to_datetime(data["날짜"]).dt.normalize()

# 1. 4월 21일까지만 학습
simulated_data = data[data["날짜"] <= cutoff_date].copy()

# 2. 4.22 ~ 4.28 실제 기온만 사용
future_temp_sim = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "avg_temp"]].copy()

# 3. 모델 실행
print(f"🕒 타임머신 작동: {cutoff_date.strftime('%Y-%m-%d')} 기준으로 학습을 시작합니다...\n")

trained_models_sim, forecast_df_sim, valid_result_sim = train_validate_forecast(
    train=simulated_data,        # ✅ preprocess_features() 넣지 않기
    forecast_days=7,
    valid_days=7,
    temp_future_df=future_temp_sim,
    manual_spikes_train=None,
    manual_spikes_future=None
)

# 4. 실제 정답 가져오기
actual_future = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "총 물량"]].copy()

actual_future.rename(columns={"총 물량": "실제_정답"}, inplace=True)
actual_future["날짜"] = actual_future["날짜"].dt.strftime("%Y-%m-%d")

# 5. 예측값과 결합
comparison_df = pd.merge(
    forecast_df_sim[["날짜", "총 물량"]],
    actual_future,
    on="날짜",
    how="left"
)

comparison_df.rename(columns={"총 물량": "모델_예측값"}, inplace=True)

# 6. 오차율 계산
comparison_df["오차율(%)"] = (
    np.abs(comparison_df["실제_정답"] - comparison_df["모델_예측값"])
    / comparison_df["실제_정답"]
    * 100
).round(2)

comparison_df["정확도(%)"] = (100 - comparison_df["오차율(%)"]).round(2)
comparison_df["차이(실제-예측)"] = comparison_df["실제_정답"] - comparison_df["모델_예측값"]

print("\n🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)")
display(comparison_df)

mean_error = comparison_df["오차율(%)"].mean()
print(f"👉 이 기간의 평균 예측 오차율은 {mean_error:.2f}% (정확도 {100 - mean_error:.2f}%) 입니다!")

🕒 타임머신 작동: 2026-04-07 기준으로 학습을 시작합니다...


=== 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 61건 / 제외 89건, full 최근 150건 중 사용 68건 / 제외 82건
지그재그_이월: train 최근 150건 중 사용 61건 / 제외 89건, full 최근 150건 중 사용 68건 / 제외 82건
기타_당일: train 최근 150건 중 사용 60건 / 제외 90건, full 최근 150건 중 사용 67건 / 제외 83건
기타_이월: train 최근 150건 중 사용 38건 / 제외 112건, full 최근 150건 중 사용 42건 / 제외 108건
총 물량: meta 최근 240건 중 사용 240건 / 제외 0건


=== 검증 성능 (MAPE) ===
총 물량: 0.0911 (정확도 90.89%)
지그재그_당일: 0.0596 (정확도 94.04%)
지그재그_이월: 0.1529 (정확도 84.71%)
기타_당일: 0.2851 (정확도 71.49%)
기타_이월: 5.9124 (정확도 -491.24%)


🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)


,날짜,모델_예측값,실제_정답,오차율(%),정확도(%),차이(실제-예측)
0,2026-04-08,19888,17288,15.04,84.96,-2600
1,2026-04-09,20645,19195,7.55,92.45,-1450
2,2026-04-10,20003,21429,6.65,93.35,1426
3,2026-04-11,14319,12137,17.98,82.02,-2182
4,2026-04-12,13301,11337,17.32,82.68,-1964
5,2026-04-13,19971,20347,1.85,98.15,376
6,2026-04-14,19787,19392,2.04,97.96,-395


👉 이 기간의 평균 예측 오차율은 9.78% (정확도 90.22%) 입니다!


In [60]:
# ==========================================================
# 🕒 백테스팅: 2026-04-21로 돌아가서 4.22 ~ 4.28 예측해보기
# ==========================================================
cutoff_date = pd.to_datetime("2026-03-31")
end_date = pd.to_datetime("2026-04-07")

data["날짜"] = pd.to_datetime(data["날짜"]).dt.normalize()

# 1. 4월 21일까지만 학습
simulated_data = data[data["날짜"] <= cutoff_date].copy()

# 2. 4.22 ~ 4.28 실제 기온만 사용
future_temp_sim = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "avg_temp"]].copy()

# 3. 모델 실행
print(f"🕒 타임머신 작동: {cutoff_date.strftime('%Y-%m-%d')} 기준으로 학습을 시작합니다...\n")

trained_models_sim, forecast_df_sim, valid_result_sim = train_validate_forecast(
    train=simulated_data,        # ✅ preprocess_features() 넣지 않기
    forecast_days=7,
    valid_days=7,
    temp_future_df=future_temp_sim,
    manual_spikes_train=None,
    manual_spikes_future=None
)

# 4. 실제 정답 가져오기
actual_future = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "총 물량"]].copy()

actual_future.rename(columns={"총 물량": "실제_정답"}, inplace=True)
actual_future["날짜"] = actual_future["날짜"].dt.strftime("%Y-%m-%d")

# 5. 예측값과 결합
comparison_df = pd.merge(
    forecast_df_sim[["날짜", "총 물량"]],
    actual_future,
    on="날짜",
    how="left"
)

comparison_df.rename(columns={"총 물량": "모델_예측값"}, inplace=True)

# 6. 오차율 계산
comparison_df["오차율(%)"] = (
    np.abs(comparison_df["실제_정답"] - comparison_df["모델_예측값"])
    / comparison_df["실제_정답"]
    * 100
).round(2)

comparison_df["정확도(%)"] = (100 - comparison_df["오차율(%)"]).round(2)
comparison_df["차이(실제-예측)"] = comparison_df["실제_정답"] - comparison_df["모델_예측값"]

print("\n🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)")
display(comparison_df)

mean_error = comparison_df["오차율(%)"].mean()
print(f"👉 이 기간의 평균 예측 오차율은 {mean_error:.2f}% (정확도 {100 - mean_error:.2f}%) 입니다!")

🕒 타임머신 작동: 2026-03-31 기준으로 학습을 시작합니다...


=== 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 54건 / 제외 96건, full 최근 150건 중 사용 61건 / 제외 89건
지그재그_이월: train 최근 150건 중 사용 54건 / 제외 96건, full 최근 150건 중 사용 61건 / 제외 89건
기타_당일: train 최근 150건 중 사용 53건 / 제외 97건, full 최근 150건 중 사용 60건 / 제외 90건
기타_이월: train 최근 150건 중 사용 33건 / 제외 117건, full 최근 150건 중 사용 38건 / 제외 112건
총 물량: meta 최근 240건 중 사용 240건 / 제외 0건


=== 검증 성능 (MAPE) ===
총 물량: 0.2110 (정확도 78.90%)
지그재그_당일: 0.2120 (정확도 78.80%)
지그재그_이월: 0.2001 (정확도 79.99%)
기타_당일: 0.1771 (정확도 82.29%)
기타_이월: 0.8652 (정확도 13.48%)


🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)


,날짜,모델_예측값,실제_정답,오차율(%),정확도(%),차이(실제-예측)
0,2026-04-01,19635,23192,15.34,84.66,3557
1,2026-04-02,19868,25429,21.87,78.13,5561
2,2026-04-03,20002,24145,17.16,82.84,4143
3,2026-04-04,15346,15320,0.17,99.83,-26
4,2026-04-05,14172,13857,2.27,97.73,-315
5,2026-04-06,19377,22620,14.34,85.66,3243
6,2026-04-07,18504,25511,27.47,72.53,7007


👉 이 기간의 평균 예측 오차율은 14.09% (정확도 85.91%) 입니다!


In [25]:
# 각 트랙별 실제값 vs 예측값 오차 확인하기
tracks = ['지그재그_당일', '지그재그_이월', '기타_당일', '기타_이월']

print("🔍 [트랙별 백테스팅 상세 오차]")
for tgt in tracks:
    # 원본 data에서 실제 정답 가져오기
    actual = data[(data['날짜'] > cutoff_date) & (data['날짜'] <= '2026-04-28')][['날짜', tgt]].copy()
    actual.rename(columns={tgt: '실제'}, inplace=True)
    actual['날짜'] = actual['날짜'].dt.strftime('%Y-%m-%d')
    
    # 예측값 가져오기
    pred = forecast_df_sim[['날짜', tgt]].copy()
    pred.rename(columns={tgt: '예측'}, inplace=True)
    
    comp = pd.merge(actual, pred, on='날짜', how='left')
    comp['차이(건)'] = comp['실제'] - comp['예측']
    
    print(f"\n▶️ {tgt}")
    display(comp)

🔍 [트랙별 백테스팅 상세 오차]

▶️ 지그재그_당일


,날짜,실제,예측,차이(건)
0,2026-04-15,7672,7527.0,145.0
1,2026-04-16,9033,7946.0,1087.0
2,2026-04-17,7369,7509.0,-140.0
3,2026-04-18,7501,5222.0,2279.0
4,2026-04-19,8641,7364.0,1277.0
5,2026-04-20,5708,6416.0,-708.0
6,2026-04-21,5794,6505.0,-711.0
7,2026-04-22,6557,NaN,NaN
8,2026-04-23,7478,NaN,NaN
9,2026-04-24,8121,NaN,NaN



▶️ 지그재그_이월


,날짜,실제,예측,차이(건)
0,2026-04-15,8539,8166.0,373.0
1,2026-04-16,9835,8919.0,916.0
2,2026-04-17,9619,8584.0,1035.0
3,2026-04-18,5881,5688.0,193.0
4,2026-04-19,4746,5045.0,-299.0
5,2026-04-20,5686,7808.0,-2122.0
6,2026-04-21,6873,8391.0,-1518.0
7,2026-04-22,7394,NaN,NaN
8,2026-04-23,8721,NaN,NaN
9,2026-04-24,9079,NaN,NaN



▶️ 기타_당일


,날짜,실제,예측,차이(건)
0,2026-04-15,4680,3126.0,1554.0
1,2026-04-16,4331,3117.0,1214.0
2,2026-04-17,3478,2809.0,669.0
3,2026-04-18,356,449.0,-93.0
4,2026-04-19,171,280.0,-109.0
5,2026-04-20,7217,5418.0,1799.0
6,2026-04-21,3693,3464.0,229.0
7,2026-04-22,3324,NaN,NaN
8,2026-04-23,3655,NaN,NaN
9,2026-04-24,3330,NaN,NaN



▶️ 기타_이월


,날짜,실제,예측,차이(건)
0,2026-04-15,9,11.0,-2.0
1,2026-04-16,10,11.0,-1.0
2,2026-04-17,11,10.0,1.0
3,2026-04-18,0,1.0,-1.0
4,2026-04-19,0,1.0,-1.0
5,2026-04-20,55,24.0,31.0
6,2026-04-21,18,11.0,7.0
7,2026-04-22,13,NaN,NaN
8,2026-04-23,7,NaN,NaN
9,2026-04-24,6,NaN,NaN
